# ⏰ Time Series Extension
Add this section to your existing notebook after the SQL Analysis block.
It adds: Trend Analysis, Demand Forecasting (ARIMA + Prophet), Dynamic Safety Stock, and Anomaly Detection.

In [ ]:
# ── Time Series Imports ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

sns.set_style('whitegrid')
print('Libraries loaded ✓')

## STEP 1 — Prepare Time Series Data
Aggregate demand by date across all products (portfolio-level) and also keep per-product series.

In [ ]:
# ── 1A. Portfolio-level daily demand time series ─────────────────────────────
# df is already loaded from your notebook with df['date'] as datetime
ts_daily = (
    df.groupby('date')['historical_demand']
      .sum()
      .sort_index()
      .rename('total_demand')
)

# Fill any date gaps with forward fill
ts_daily = ts_daily.asfreq('D').fillna(method='ffill')

print(f'Date range : {ts_daily.index.min()} → {ts_daily.index.max()}')
print(f'Total days : {len(ts_daily)}')
ts_daily.head()

In [ ]:
# ── 1B. Per-product time series (pick top 5 by total demand) ─────────────────
top5_products = (
    df.groupby('product_id')['historical_demand']
      .sum()
      .nlargest(5)
      .index.tolist()
)

ts_products = (
    df[df['product_id'].isin(top5_products)]
      .groupby(['date', 'product_id'])['historical_demand']
      .sum()
      .unstack('product_id')
      .sort_index()
      .fillna(method='ffill')
)

print('Top 5 products:', top5_products)
ts_products.head()

## STEP 2 — Visualise Demand Over Time

In [ ]:
# ── 2A. Portfolio demand trend with 7-day rolling average ───────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ts_daily.index, ts_daily.values, alpha=0.4, color='steelblue', label='Daily Demand')
ax.plot(ts_daily.index, ts_daily.rolling(7).mean(), color='navy', lw=2, label='7-Day Rolling Avg')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
plt.title('Portfolio-Level Daily Demand Over Time', fontsize=14)
plt.ylabel('Total Demand')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 2B. Per-product demand trends ───────────────────────────────────────────
fig, axes = plt.subplots(len(top5_products), 1, figsize=(14, 3 * len(top5_products)), sharex=True)
colors = sns.color_palette('tab10', len(top5_products))

for ax, pid, color in zip(axes, top5_products, colors):
    series = ts_products[pid]
    ax.plot(series.index, series.values, color=color, alpha=0.5)
    ax.plot(series.index, series.rolling(7).mean(), color=color, lw=2)
    ax.set_title(f'Product {pid}', fontsize=11)
    ax.set_ylabel('Demand')

plt.suptitle('Top 5 Products — Daily Demand Trends', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## STEP 3 — Stationarity Test (ADF)
ARIMA requires a stationary series. We test and difference if needed.

In [ ]:
# ── 3. Augmented Dickey-Fuller test ─────────────────────────────────────────
def adf_test(series, name='Series'):
    result = adfuller(series.dropna())
    p = result[1]
    stationary = p < 0.05
    print(f'{name}: ADF stat={result[0]:.4f}, p-value={p:.4f}  →  {"✅ Stationary" if stationary else "❌ Non-stationary (needs differencing)"}')
    return stationary

is_stationary = adf_test(ts_daily, 'Portfolio Demand')

if not is_stationary:
    ts_diff = ts_daily.diff().dropna()
    adf_test(ts_diff, 'Portfolio Demand (differenced)')
    d_order = 1
else:
    d_order = 0

print(f'\nRecommended d order for ARIMA: {d_order}')

## STEP 4 — Seasonal Decomposition
Breaks demand into Trend + Seasonality + Residual components.

In [ ]:
# ── 4. Decompose portfolio demand ────────────────────────────────────────────
# Use period=7 for weekly seasonality; change to 30 for monthly
decomp = seasonal_decompose(ts_daily, model='additive', period=7, extrapolate_trend='freq')

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
components = [
    (decomp.observed,  'Observed',   'steelblue'),
    (decomp.trend,     'Trend',      'darkorange'),
    (decomp.seasonal,  'Seasonality','green'),
    (decomp.resid,     'Residual',   'red'),
]
for ax, (data, title, color) in zip(axes, components):
    ax.plot(data.index, data.values, color=color)
    ax.set_title(title, fontsize=11)

plt.suptitle('Seasonal Decomposition of Portfolio Demand', fontsize=14)
plt.tight_layout()
plt.show()

# Seasonality strength score (0-1)
seas_strength = 1 - (decomp.resid.var() / (decomp.seasonal + decomp.resid).var())
print(f'\nSeasonality Strength Score: {seas_strength:.2f}  (>0.6 = strong seasonal pattern)')

## STEP 5 — ARIMA Demand Forecast
Fit ARIMA on portfolio demand and forecast the next 30 days.

In [ ]:
# ── 5A. ACF / PACF plots to visually pick p and q ───────────────────────────
series_for_acf = ts_daily.diff().dropna() if d_order == 1 else ts_daily

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(series_for_acf,  lags=30, ax=axes[0], title='ACF  (helps pick q)')
plot_pacf(series_for_acf, lags=30, ax=axes[1], title='PACF (helps pick p)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5B. Fit ARIMA and forecast 30 days ──────────────────────────────────────
FORECAST_DAYS = 30
TRAIN_SIZE = int(len(ts_daily) * 0.85)

train = ts_daily.iloc[:TRAIN_SIZE]
test  = ts_daily.iloc[TRAIN_SIZE:]

# Auto-select order: (1,d,1) is a sensible default; tune p,q via ACF/PACF above
arima_order = (1, d_order, 1)
model = ARIMA(train, order=arima_order)
fitted = model.fit()

print(fitted.summary())

In [ ]:
# ── 5C. Forecast and plot ────────────────────────────────────────────────────
forecast_obj = fitted.get_forecast(steps=len(test) + FORECAST_DAYS)
fc_mean = forecast_obj.predicted_mean
fc_ci   = forecast_obj.conf_int(alpha=0.05)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index, train.values, color='steelblue', label='Train')
ax.plot(test.index,  test.values,  color='orange',    label='Actual (test)')
ax.plot(fc_mean.index, fc_mean.values, color='green', lw=2, linestyle='--', label='Forecast')
ax.fill_between(fc_ci.index, fc_ci.iloc[:, 0], fc_ci.iloc[:, 1],
                color='green', alpha=0.15, label='95% CI')
ax.axvline(train.index[-1], color='gray', linestyle=':', label='Train/Test Split')
plt.title(f'ARIMA{arima_order} — Demand Forecast (+{FORECAST_DAYS} days)', fontsize=14)
plt.ylabel('Total Demand')
plt.legend()
plt.tight_layout()
plt.show()

# MAPE on test set
test_aligned = test.reindex(fc_mean.index[:len(test)])
mape = np.mean(np.abs((test_aligned - fc_mean[:len(test)]) / test_aligned.replace(0, np.nan))) * 100
print(f'\nARIMA Test MAPE: {mape:.2f}%  (lower is better)')

## STEP 6 — Prophet Forecast (per product)
Prophet handles seasonality + holidays automatically and works well per-product.

In [ ]:
# ── 6. Prophet forecast for each of the top 5 products ─────────────────────
try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except ImportError:
    print('Prophet not installed. Run: pip install prophet')
    PROPHET_AVAILABLE = False

if PROPHET_AVAILABLE:
    FORECAST_HORIZON = 30  # days ahead
    prophet_forecasts = {}

    fig, axes = plt.subplots(len(top5_products), 1,
                             figsize=(14, 4 * len(top5_products)), sharex=False)

    for ax, pid in zip(axes, top5_products):
        series = ts_products[pid].reset_index()
        series.columns = ['ds', 'y']
        series = series.dropna()

        m = Prophet(daily_seasonality=True, weekly_seasonality=True,
                    yearly_seasonality=False, interval_width=0.95)
        m.fit(series)

        future = m.make_future_dataframe(periods=FORECAST_HORIZON)
        forecast = m.predict(future)
        prophet_forecasts[pid] = forecast

        # Plot
        ax.plot(series['ds'], series['y'], color='steelblue', alpha=0.5, label='Historical')
        ax.plot(forecast['ds'], forecast['yhat'], color='green', lw=2, label='Forecast')
        ax.fill_between(forecast['ds'], forecast['yhat_lower'], forecast['yhat_upper'],
                        alpha=0.15, color='green', label='95% CI')
        ax.axvline(series['ds'].max(), color='gray', linestyle=':')
        ax.set_title(f'Product {pid} — Prophet Forecast', fontsize=11)
        ax.set_ylabel('Demand')
        ax.legend(fontsize=8)

    plt.suptitle('Prophet Demand Forecasts — Top 5 Products', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

## STEP 7 — Dynamic Safety Stock
Replace the static formula with a rolling-window version that adapts to recent demand variability.

In [ ]:
# ── 7. Dynamic Safety Stock using rolling std ────────────────────────────────
Z = 1.65   # 95% service level
WINDOW = 14  # 14-day rolling window

# Use top product as example
example_pid = top5_products[0]
prod_series = ts_products[example_pid].copy()

rolling_std  = prod_series.rolling(WINDOW).std()
avg_lead_time = df[df['product_id'] == example_pid]['lead_time_days'].mean()

dynamic_ss = Z * rolling_std * np.sqrt(avg_lead_time)

# Static benchmark from your existing formula
static_variability = df[df['product_id'] == example_pid]['demand_variability'].mean()
static_demand      = df[df['product_id'] == example_pid]['historical_demand'].mean()
static_ss_value    = static_variability * static_demand * np.sqrt(avg_lead_time)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(dynamic_ss.index, dynamic_ss.values, color='darkorange', lw=2, label='Dynamic Safety Stock')
ax.axhline(static_ss_value, color='steelblue', linestyle='--', lw=2, label=f'Static Safety Stock ({static_ss_value:.1f})')
plt.title(f'Product {example_pid} — Dynamic vs Static Safety Stock', fontsize=13)
plt.ylabel('Safety Stock Units')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Static Safety Stock : {static_ss_value:.1f} units (fixed)')
print(f'Dynamic range       : {dynamic_ss.min():.1f} – {dynamic_ss.max():.1f} units (adapts to demand shifts)')

## STEP 8 — Dynamic EOQ Using Forecasted Demand
Replace `annual_demand = daily_demand * 365` with the ARIMA/Prophet forecast.

In [ ]:
# ── 8. Dynamic EOQ using next-30-day forecast ────────────────────────────────
dynamic_eoq_rows = []

for pid in top5_products:
    # Use Prophet forecast if available, else ARIMA portfolio avg
    if PROPHET_AVAILABLE and pid in prophet_forecasts:
        fc = prophet_forecasts[pid]
        last_date = ts_products[pid].index.max()
        future_fc = fc[fc['ds'] > last_date]['yhat'].clip(lower=0)
        forecasted_daily = future_fc.mean()
    else:
        forecasted_daily = ts_products[pid].rolling(30).mean().iloc[-1]

    forecasted_annual = forecasted_daily * 365
    avg_ordering = df[df['product_id'] == pid]['ordering_cost'].mean()
    avg_holding  = df[df['product_id'] == pid]['holding_cost'].mean()

    static_annual = df[df['product_id'] == pid]['historical_demand'].mean() * 365
    eoq_static   = np.sqrt(2 * static_annual    * avg_ordering / avg_holding)
    eoq_dynamic  = np.sqrt(2 * forecasted_annual * avg_ordering / avg_holding)

    dynamic_eoq_rows.append({
        'product_id'        : pid,
        'forecasted_daily'  : round(forecasted_daily, 2),
        'forecasted_annual' : round(forecasted_annual, 2),
        'EOQ_static'        : round(eoq_static, 2),
        'EOQ_dynamic'       : round(eoq_dynamic, 2),
        'EOQ_change_%'      : round((eoq_dynamic - eoq_static) / eoq_static * 100, 2)
    })

eoq_df = pd.DataFrame(dynamic_eoq_rows)
print(eoq_df.to_string(index=False))

## STEP 9 — Anomaly Detection on Demand
Use rolling z-scores to flag demand spikes and drops automatically.

In [ ]:
# ── 9. Rolling Z-score anomaly detection ────────────────────────────────────
WINDOW_ANOM = 14
Z_THRESHOLD = 2.5

rolling_mean = ts_daily.rolling(WINDOW_ANOM).mean()
rolling_std  = ts_daily.rolling(WINDOW_ANOM).std()
z_score      = (ts_daily - rolling_mean) / rolling_std

anomalies = ts_daily[z_score.abs() > Z_THRESHOLD]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Top: demand with anomaly flags
axes[0].plot(ts_daily.index, ts_daily.values, color='steelblue', alpha=0.6, label='Demand')
axes[0].scatter(anomalies.index, anomalies.values, color='red', s=60, zorder=5, label='Anomaly')
axes[0].set_title('Demand Anomalies (Rolling Z-Score)', fontsize=13)
axes[0].set_ylabel('Total Demand')
axes[0].legend()

# Bottom: z-score series
axes[1].plot(z_score.index, z_score.values, color='purple', alpha=0.7)
axes[1].axhline( Z_THRESHOLD, color='red', linestyle='--', lw=1.5, label=f'+{Z_THRESHOLD}σ')
axes[1].axhline(-Z_THRESHOLD, color='red', linestyle='--', lw=1.5, label=f'-{Z_THRESHOLD}σ')
axes[1].set_title('Z-Score', fontsize=11)
axes[1].set_ylabel('Z-Score')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Anomalies detected: {len(anomalies)}')
print(anomalies)

## STEP 10 — Summary: Time Series Insights
Consolidate all findings.

In [ ]:
# ── 10. Print consolidated summary ──────────────────────────────────────────
print('=' * 60)
print('      TIME SERIES ANALYSIS — KEY FINDINGS')
print('=' * 60)

print(f"\n📈 Stationarity  : {'Stationary' if d_order == 0 else 'Non-stationary → differenced (d=1)'}")
print(f"🔄 Seasonality   : Strength = {seas_strength:.2f}  {'(significant)' if seas_strength > 0.6 else '(weak)'}")
print(f"🎯 ARIMA MAPE    : {mape:.2f}%")
print(f"⚠️  Anomalies     : {len(anomalies)} demand spikes/drops detected")
print(f"\n📦 Dynamic Safety Stock vs Static (Product {example_pid}):")
print(f"   Static  : {static_ss_value:.1f} units")
print(f"   Dynamic : {dynamic_ss.mean():.1f} units (avg), range {dynamic_ss.min():.1f}–{dynamic_ss.max():.1f}")
print(f"\n🔢 Dynamic EOQ Changes:")
for _, row in eoq_df.iterrows():
    direction = '▲' if row['EOQ_change_%'] > 0 else '▼'
    print(f"   Product {row['product_id']}: {direction} {abs(row['EOQ_change_%'])}% vs static EOQ")
print('=' * 60)